# Model owner

You hold a private LoRA adapter for gemma-4-31B-it. This notebook verifies the enclave, uploads the adapter over the attestation-pinned channel, approves the run and collects the receipt. You never see the benchmark owner's prompts or the results.

Setup once: `pip install git+https://github.com/tinfoilsh/double-blind-eval` and the [`tinfoil` CLI](https://docs.tinfoil.sh/containers/cli). Create your key with `dbe keygen --party model-owner` and have the operator put the printed public key into `tinfoil-config.yml`.

In [ ]:
PARTY = "model-owner"
import os, subprocess, json
ENCLAVE = os.environ.get("DBE_ENCLAVE", "dbe.tinfoil.containers.tinfoil.dev")
REPO = os.environ.get("DBE_REPO", "tinfoilsh/double-blind-eval")
TAG = os.environ.get("DBE_TAG", "v0.1.0")
os.environ.update(DBE_ENCLAVE=ENCLAVE, DBE_REPO=REPO, DBE_PARTY=PARTY)

def dbe(*args):
    """Run a dbe command and print its output."""
    proc = subprocess.run(["dbe", *args], capture_output=True, text=True)
    print(proc.stdout or proc.stderr)
    return proc

## 1. Verify the enclave

`dbe verify` checks the Sigstore-published measurement of the release against the enclave's live hardware attestation and pins the TLS key. Everything below refuses to talk to anything else.

In [ ]:
dbe("verify")

## 2. Upload the private adapter

A PEFT adapter directory (`adapter_config.json` + `adapter_model.safetensors`) or a `.tar.gz` of one. It is held in enclave memory and loaded into vLLM; it never touches the host disk.

In [ ]:
ADAPTER_PATH = os.environ.get("DBE_ADAPTER", "./adapter")
dbe("model", "upload", ADAPTER_PATH)

## 3. Review and approve the run manifest

The manifest names the adapter hash, the benchmark hash, the sampling parameters and the output policy. Compare `manifest_sha256` with the benchmark owner out of band, then sign it. The run starts when both parties have approved.

In [ ]:
dbe("manifest")

In [ ]:
dbe("approve")

## 4. Wait for the run and collect the receipt

The output policy gives the model owner the receipt only: proof of what ran, signed by the enclave.

In [ ]:
dbe("run", "--wait")
dbe("receipt", "get", "--out", "receipt.json")
dbe("receipt", "verify", "receipt.json", "--tag", TAG)